# Q1. Install MLflow

In [25]:
!mlflow --version

mlflow, version 3.10.1


# Q2. Download and preprocess the data

In [10]:
import pandas as pd

In [3]:
!python preprocess_data.py --raw_data_path ./data/ --dest_path ./output

# Q3. Train a model with autolog

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [28]:
import mlflow
# mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment("Taxi_Lesson2")
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.sklearn.autolog()

In [29]:
# Load Dataset 
train = pd.read_pickle("./output/train.pkl")
test = pd.read_pickle("./output/train.pkl")
val = pd.read_pickle("./output/val.pkl")

In [30]:
# Training model
with mlflow.start_run():

    mlflow.set_tag("developer", "Valentin")
    
    mlflow.log_param("train-data-path", "./data/green_tripdata_2023-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2023-02.csv")

    reg = RandomForestRegressor()
    reg.fit(train[0],train[1])

    y_pred = reg.predict(val[0])
    rmse = mean_squared_error(val[1], y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

2026/03/17 14:39:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run spiffy-dolphin-665 at: http://localhost:5000/#/experiments/2/runs/d7ed5fc067bb49ea84e16db6a05fc677
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [31]:
print(mlflow.get_tracking_uri())

http://localhost:5000


In [32]:
client = mlflow.tracking.MlflowClient()

'ID:2 | Nom: Taxi_Lesson2'

runs = client.search_runs(experiment_ids=['2'])
for r in runs : 
    print(f'Run:{r.info.run_id}')

Run:d7ed5fc067bb49ea84e16db6a05fc677
Run:4ecd9af694d34b97a855fc0bfd1f96b1
Run:e9f5f70ebbd54b4abd6051630609825d
Run:155ccdf922a64108bef886a190536c82


In [36]:
run = client.get_run('d7ed5fc067bb49ea84e16db6a05fc677').data.params
run['min_samples_split']

'2'

# Q4. Launch the tracking server locally

In [38]:
print(f"Tracking URI: '{mlflow.get_tracking_uri()}'")

Tracking URI: 'http://localhost:5000'


In [40]:
mlflow.search_experiments()

[<Experiment: artifact_location='/home/valentin/MlOps-Zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1773748107252, experiment_id='2', last_update_time=1773748107252, lifecycle_stage='active', name='Taxi_Lesson2', tags={}, workspace='default'>,
 <Experiment: artifact_location='/home/drkuzz/Documents/MlOps-Zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1773332786450, experiment_id='1', last_update_time=1773332786450, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1773332052292, experiment_id='0', last_update_time=1773332052292, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [41]:
print("In addition to backend-store-uri, we need default-artifact-root ")

In addition to backend-store-uri, we need default-artifact-root 


# Q5. Tune model hyperparameters